In [1]:
# ============================================
# Forecast & Inventory Risk Analytics
# Incremental Monthly Update Generator
# Author: Diego Loera
# Purpose: Generate one new month to UPSERT into PostgreSQL
# Output: stg_fact_sales_month.csv, stg_fact_forecast_month.csv, stg_fact_inventory_month.csv
# ============================================

import numpy as np
import pandas as pd
from pathlib import Path

SEED = 123
rng = np.random.default_rng(SEED)

OUT_DIR = Path("./stg_dataset")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- 1) Load existing dim_product (same SKUs) ---
dim_product = pd.read_csv("dim_product.csv")  # upload it to Colab or point to correct path
dim_product.head()

,product_id,product_name,product_family,unit_cost_mxn
0,P001,Part_P001,Interior,158.69
1,P002,Part_P002,Fasteners,47.47
2,P003,Part_P003,Trim,172.04
3,P004,Part_P004,Fasteners,24.06
4,P005,Part_P005,Electrical,369.68


In [2]:
# --- 2) Configuration for the new month ---
new_month = pd.Timestamp("2026-01-01")  # change to any month-start date

families = dim_product["product_family"].unique().tolist()

# Use the same ranges as historical (keep consistent)
cost_ranges = {
    "Electrical": (120, 420),
    "Fasteners": (10, 55),
    "Interior": (80, 260),
    "Trim": (60, 220),
}

demand_ranges = {
    "Electrical": (900, 4500),
    "Fasteners": (1500, 8000),
    "Interior": (600, 3000),
    "Trim": (700, 3500),
}

def seasonal_factor(month_idx: int) -> float:
    return 1.0 + 0.15 * np.sin(2 * np.pi * (month_idx / 12.0))

def clamp_int(x):
    return int(max(0, np.round(x)))

def sku_volatility(family: str) -> float:
    if family == "Fasteners":
        return 0.20
    if family == "Electrical":
        return 0.18
    if family == "Trim":
        return 0.16
    return 0.14

# "Learned" stable parameters per SKU (we simulate it here deterministically)
# In a real case, you'd load prior params from a table; for portfolio it’s enough.
sku_params = {}
for _, r in dim_product.iterrows():
    pid = r["product_id"]
    fam = r["product_family"]
    # deterministic-ish baseline per SKU
    base = rng.uniform(*demand_ranges[fam])
    vol = sku_volatility(fam)
    bias = rng.normal(0.0, 0.04)
    sku_params[pid] = {"family": fam, "base_demand": base, "vol": vol, "bias": bias}

# Control difficulty and DOI targets
sku_difficulty = {pid: rng.uniform(0.08, 0.22) for pid in dim_product["product_id"]}
sku_base_doi   = {pid: rng.uniform(35, 65) for pid in dim_product["product_id"]}

m_idx = new_month.month - 1
s = seasonal_factor(m_idx)
trend = 1.0  # for incremental example, keep trend neutral

In [3]:
# --- 3) Generate month data (sales, forecast, inventory) ---
rows_sales = []
rows_forecast = []
rows_inventory = []

for pid in dim_product["product_id"]:
    p = sku_params[pid]
    base = p["base_demand"]

    # Actual demand
    noise = rng.normal(1.0, p["vol"])
    actual = base * s * trend * noise
    actual_units = clamp_int(actual)

    # Forecast
    amp = sku_difficulty[pid]
    e = rng.normal(loc=p["bias"], scale=amp)
    forecast_units = clamp_int(actual_units * (1.0 + e))

    # Inventory (target DOI 30–70)
    doi = max(5, rng.normal(sku_base_doi[pid], 12))
    inv_units = actual_units * (doi / 30.0)
    if rng.random() < 0.08:
        inv_units *= rng.uniform(1.2, 1.7)
    inv_units = clamp_int(inv_units)

    rows_sales.append([new_month.date(), pid, actual_units])
    rows_forecast.append([new_month.date(), pid, forecast_units])
    rows_inventory.append([new_month.date(), pid, inv_units])

stg_sales = pd.DataFrame(rows_sales, columns=["month", "product_id", "actual_units_sold"])
stg_forecast = pd.DataFrame(rows_forecast, columns=["month", "product_id", "forecast_units"])
stg_inventory = pd.DataFrame(rows_inventory, columns=["month", "product_id", "ending_inventory_units"])

stg_sales.head()

,month,product_id,actual_units_sold
0,2026-01-01,P001,2210
1,2026-01-01,P002,1800
2,2026-01-01,P003,1382
3,2026-01-01,P004,5269
4,2026-01-01,P005,4132


In [4]:
# --- 4) Quick sanity checks ---
tmp = stg_sales.merge(stg_forecast, on=["month","product_id"]).merge(stg_inventory, on=["month","product_id"])
tmp["abs_error"] = (tmp["actual_units_sold"] - tmp["forecast_units"]).abs()

weighted_mape = tmp["abs_error"].sum() / max(tmp["actual_units_sold"].sum(), 1)
avg_doi = np.mean(np.where(tmp["actual_units_sold"] == 0, np.nan, (tmp["ending_inventory_units"]/tmp["actual_units_sold"]) * 30))

weighted_mape, avg_doi

(np.float64(0.10846417463666003), np.float64(50.51697429870503))

In [5]:
# --- 5) Export staging CSVs for PostgreSQL UPSERT ---
stg_sales.to_csv(OUT_DIR / "stg_fact_sales_month.csv", index=False)
stg_forecast.to_csv(OUT_DIR / "stg_fact_forecast_month.csv", index=False)
stg_inventory.to_csv(OUT_DIR / "stg_fact_inventory_month.csv", index=False)

print("Saved:", list(OUT_DIR.glob("*.csv")))
print("Month:", new_month.date())

Saved: [PosixPath('stg_dataset/stg_fact_sales_month.csv'), PosixPath('stg_dataset/stg_fact_forecast_month.csv'), PosixPath('stg_dataset/stg_fact_inventory_month.csv')]
Month: 2026-01-01


In [6]:
#Create zip with all final files
import shutil
from google.colab import files

folder_to_zip = "stg_dataset"

zip_filename = "project_stg_dataset"
shutil.make_archive(zip_filename, 'zip', folder_to_zip)

print("ZIP created successfully!")

ZIP created successfully!


In [7]:
# Download dataset
files.download(f"{zip_filename}.zip")

print("Dataset downloaded successfully")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Dataset downloaded successfully
